# Annotate against your own reference

Run vX on a four-row AIRR cohort and two wide paired queries using the bundled synthetic VDJdb-format TSV. Open this notebook from the repository root or the complete examples directory. The commands and the fixture require no download.

For your own complete pMHC reference, preserve the receptor species, locus, junction and V/J calls, and the peptide, MHC chains and class. Use a shared `complex_id` for genuine paired reference chains. Pass the table or archive to `--vdjdb`. This fixture has no matched null bank, so its selected scores are descriptive evidence.

In [ ]:
# Identify the installed CLI and print the fresh kernel environment.
import os, sys, subprocess, tempfile, json
from pathlib import Path
from importlib.metadata import version
os.environ["POLARS_MAX_THREADS"] = "1"
print("Python:", sys.version, "CLI interpreter:", sys.executable)
for package in ("vdjmatch", "vdjtools", "seqtree", "polars"):
    print(package, version(package))
# Open this notebook from the repository root or the examples directory.
examples = next((p for p in (Path.cwd()/"examples", Path.cwd())
                 if (p/"inputs/reference.tsv").is_file()), None)
if examples is None:
    raise RuntimeError("Use the downloaded complete examples bundle, not this notebook alone")
# Deterministic demonstration: no sampling or RNG seeds are used.
output = Path(tempfile.mkdtemp(prefix="vdjmatch-example-"))/"fresh"
output.mkdir()
print("Fresh output directory:", output)


In [ ]:
# Run raw inputs with this kernel's installed CLI, on any supported platform.
subprocess.run([sys.executable, "-m", "vdjmatch.cli", "annotate",
                "--metadata", str(examples/"inputs/samples.tsv"),
                "--vdjdb", str(examples/"inputs/reference.tsv"), "--threads", "4",
                "--output-prefix", str(output/"cohort")], check=True, timeout=120)
subprocess.run([sys.executable, "-m", "vdjmatch.cli", "annotate",
                str(examples/"inputs/paired.tsv"), "--locus", "paired",
                "--vdjdb", str(examples/"inputs/reference.tsv"), "--threads", "4",
                "--output-prefix", str(output/"paired")], check=True, timeout=120)

In [ ]:
# Inspect source identities, vX dispositions, and the paired marginal output.
import polars as pl

def table(prefix, kind):
    return pl.read_csv(output / f"{prefix}.{kind}.tsv", separator="	")

for prefix, count in (("cohort", 4), ("paired", 2)):
    queries = table(prefix, "queries")
    selections = table(prefix, "selection")
    manifest = json.loads((output / f"{prefix}.manifest.json").read_text())
    assert queries.height == selections.height == count
    assert manifest["score_models"] == ["vx"]
    assert manifest["exact_hits_excluded"] is False
    assert manifest["reference"]["input_sha256"]
    assert "calibration" not in manifest
    source_key = "source_pair_id" if prefix == "paired" else "source_sequence_id"
    print(prefix, queries.select("sample_id", source_key, "status"))
    print(selections.select("selected_epitope", "score", "selection_status"))

cohort = table("cohort", "selection")
assert cohort.select("sample_id", "source_sequence_id").n_unique() == 4
assert cohort["source_sequence_id"].n_unique() == 2
assert table("paired", "queries")["status"].to_list() == ["matched", "invalid_beta_query"]
marginal = table("paired", "TRA.selection").filter(pl.col("source_pair_id") == "pair2")
assert marginal.height == 1
assert marginal["selected_epitope"].to_list() == ["DEMO_B"]
print(marginal.select("selected_epitope", "score", "selection_status"))

The same source ID can occur in two samples. Its sample ID distinguishes the observations. The incomplete pair remains in the paired table and its eligible α chain has separate marginal evidence. `DEMO_A` and `DEMO_B` are fixture labels. A selected target in this example does not measure accuracy or establish an error rate.

## Use labels without MHC metadata

For a reference with opaque class labels, use the direct labeled-reference API. Each row here contains a junction with its conserved anchors and the supplied label. This workflow reports match evidence without vX pMHC selection or null-bank calibration.

In [ ]:
# Look up three queries against a custom two-label reference.
from vdjmatch import Annotator

reference = [("CASSIRSSYEQYF", "class-a"), ("CASSLGQETQYF", "class-b")]
annotator = Annotator.from_labels(reference, locus="TRB")
calls = annotator.annotate(["CASSIRSSYEQYF", "CASSLGQETQYF", "CWWWWWWWF"])
assert calls["vdjmatch_epitope"].to_list() == ["class-a", "class-b", None]
print(calls.select("vdjmatch_epitope"))